# Approach 1: a one-stage CNN detector (YOLOv8s)

ENPM818Z, Lecture 4. YOLOv8s is GP2's first model: a convolutional network that
predicts a box at every cell of three grids, then removes the duplicates with
non-maximum suppression (NMS). This notebook follows the slides step by step, on the
same photo, and lets you check every number yourself.

1. Run the detector.
2. Open its raw output: one matrix, one column per grid cell.
3. Decode one cell into a box by hand.
4. Apply the confidence cut and NMS yourself, and compare with Ultralytics.
5. Sweep the confidence cut.
6. Time it.

**Requirements:** `pip install ultralytics` (it installs PyTorch, NumPy and
matplotlib). A GPU is optional: on a CPU every step still works, only slower. The
weights (`yolov8s.pt`, 22 MB) download on the first run.

In [ ]:
# This original version of the code is from fall 2025 and it was modified and
# improved by Anthropic Claude Opus 5.5.
# Signed: Zeid Kootbally

import os
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import ultralytics

# The photo from the slides. To use a CARLA frame from GP1, put its path here.
IMAGE = os.path.join(os.path.dirname(ultralytics.__file__), "assets", "bus.jpg")
photo = np.asarray(Image.open(IMAGE).convert("RGB"))
H, W = photo.shape[:2]
print(f"image: {W} x {H} pixels (width x height)")


def draw(ax, image, boxes, names, title):
    """boxes: list of ((x0, y0, x1, y1), class_id, score) in image pixels."""
    ax.imshow(image)
    ax.set_axis_off()
    ax.set_title(title)
    for (x0, y0, x1, y1), c, s in boxes:
        ax.add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=2,
                                        ec="tab:orange"))
        ax.text(x0, y0, f"{names[c]} {s:.2f}", color="white", fontsize=8, va="bottom",
                bbox=dict(fc="tab:orange", ec="none", pad=1))


def iou(a, b):
    """Intersection over union of two boxes given as (x0, y0, x1, y1)."""
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter)


def latency_ms(model, n=50, warmup=10):
    """Median time of the whole predict call (file in, boxes out), batch 1."""
    for _ in range(warmup):
        model(IMAGE, verbose=False)
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        model(IMAGE, verbose=False)
        times.append(1000 * (time.perf_counter() - t0))
    return float(np.median(times))

from ultralytics import YOLO

model = YOLO("yolov8s.pt")        # COCO-pretrained, 80 classes

## 1. Run the detector

One call returns the final detections. Each has a class, a confidence between 0 and 1,
and a box from its top-left corner (x0, y0) to its bottom-right corner (x1, y1), in
pixels of the photo, x to the right and y down.

In [ ]:
result = model(IMAGE, verbose=False)[0]
final = [(b.tolist(), int(c), float(s))
         for b, c, s in zip(result.boxes.xyxy, result.boxes.cls, result.boxes.conf)]
for (x0, y0, x1, y1), c, s in final:
    print(f"{result.names[c]:>10s} {s:.3f}  ({x0:.1f}, {y0:.1f}) to ({x1:.1f}, {y1:.1f})")

fig, ax = plt.subplots(figsize=(5, 6.5))
draw(ax, photo, final, result.names, f"YOLOv8s: {len(final)} objects")
plt.show()

## 2. The raw output: one matrix

The network does not output a list of objects. Ultralytics first resizes the photo so
its long side is 640 pixels. The head then runs at every cell of three grids (strides 8,
16 and 32) and gives **144 raw numbers per cell**: 64 for the box (4 sides x 16 bins)
and 80 class scores. The model decodes them to **84 per cell**: the box center and size
(cx, cy, w, h) in pixels of the resized input, then 80 class scores between 0 and 1.

In [ ]:
x = model.predictor.preprocess([photo[:, :, ::-1].copy()])   # the model's own resize (it expects BGR)
print("input tensor:", tuple(x.shape), " (batch, colors, height, width)")
with torch.no_grad():
    out = model.predictor.model(x)
decoded = out[0][0].cpu().numpy()          # (84, cells)
raw = [r.cpu() for r in out[1]]            # per stride: (1, 144, rows, cols)

cells = 0
for r, stride in zip(raw, (8, 16, 32)):
    rows, cols = r.shape[2], r.shape[3]
    cells += rows * cols
    print(f"stride {stride:2d}: {cols} x {rows} = {rows * cols:5d} cells, {r.shape[1]} raw numbers each")
print(f"total: {cells} cells")
print("decoded matrix:", decoded.shape, " (84 rows, one column per cell)")

## 3. Decode one cell by hand

Take the most confident cell. For each side of its box the head gives 16 raw scores. A
**softmax** turns them into 16 probabilities (positive, adding to 1) for the distance
0, 1, ..., 15, in strides. The distance is their weighted average, times the stride.
The box corners are the cell center minus the left and top distances, plus the right
and bottom ones. The class score is the raw class number passed through a **sigmoid**,
1 / (1 + e^-x).

In [ ]:
col = int(decoded[4:].max(0).argmax())          # the most confident cell
cls_id = int(decoded[4:, col].argmax())
print(f"cell column {col}: {result.names[cls_id]} {decoded[4 + cls_id, col]:.3f}")

start = 0                                       # which grid, which cell
for r, stride in zip(raw, (8, 16, 32)):
    rows, cols = r.shape[2], r.shape[3]
    if col < start + rows * cols:
        row, c = divmod(col - start, cols)
        break
    start += rows * cols
cx0, cy0 = (c + 0.5) * stride, (row + 0.5) * stride
print(f"stride {stride}, column {c}, row {row}: cell center ({cx0}, {cy0}) in input pixels")

p = torch.softmax(r[0, :64, row, c].reshape(4, 16), dim=1)   # 16 probabilities per side
bins = (p * torch.arange(16.0)).sum(dim=1).tolist()          # weighted average, in strides
left, top, right, bottom = [b * stride for b in bins]
print("weighted averages (left, top, right, bottom):", [round(b, 3) for b in bins])
print(f"box by hand:     ({cx0 - left:.1f}, {cy0 - top:.1f}) to ({cx0 + right:.1f}, {cy0 + bottom:.1f})")
cx, cy, w, h = decoded[:4, col]
print(f"model's decode:  ({cx - w / 2:.1f}, {cy - h / 2:.1f}) to ({cx + w / 2:.1f}, {cy + h / 2:.1f})")
logit = float(r[0, 64 + cls_id, row, c])
print(f"class score: sigmoid({logit:.3f}) = {1 / (1 + np.exp(-logit)):.3f}")

fig, ax = plt.subplots(figsize=(6, 2.8))
ax.bar(range(16), p[1].numpy())
ax.axvline(bins[1], color="tab:orange", ls="--", label=f"weighted average {bins[1]:.3f}")
ax.set_xlabel("distance to the top side, in strides")
ax.set_ylabel("probability")
ax.legend()
plt.show()

## 4. The confidence cut and NMS, by hand

Keep the cells whose best class score is above 0.25 (the Ultralytics default). Then
**NMS**, per class: sort by score, keep the best box, delete every box of the same class
that overlaps it with an IoU above 0.7 (the Ultralytics default), and repeat.

In [ ]:
def nms(boxes, iou_cut=0.7, per_class=True):
    """boxes: list of (corners, class_id, score). Returns the kept boxes."""
    boxes = sorted(boxes, key=lambda b: -b[2])
    kept = []
    while boxes:
        best = boxes.pop(0)
        kept.append(best)
        boxes = [b for b in boxes
                 if (per_class and b[1] != best[1]) or iou(b[0], best[0]) <= iou_cut]
    return kept


conf_cut = 0.25
candidates = []
for col in np.where(decoded[4:].max(0) > conf_cut)[0]:
    cx, cy, w, h = decoded[:4, col]
    c = int(decoded[4:, col].argmax())
    candidates.append(([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], c, float(decoded[4 + c, col])))

kept = nms(candidates)
print(f"{decoded.shape[1]} cells -> {len(candidates)} above {conf_cut} -> {len(kept)} after NMS")
print(f"Ultralytics: {len(result.boxes)} objects")
for (x0, y0, x1, y1), c, s in kept:
    print(f"  {result.names[c]:>10s} {s:.3f}")

## 5. Sweep the confidence cut

The same model, four cuts. On a car you choose the cut per model, on validation data.

In [ ]:
for cut in (0.10, 0.25, 0.50, 0.75):
    r = model(IMAGE, conf=cut, verbose=False)[0]
    names = [r.names[int(c)] for c in r.boxes.cls]
    print(f"cut {cut:.2f}: {len(names):2d} objects  {sorted(set(names))}")

## 6. Time

The median of 50 runs after 10 warm-up runs. It times the whole call: reading the file,
resizing, the network and NMS, one image at a time. At 20 Hz a camera frame arrives
every 50 ms, and the detector shares that time with the rest of the car.

In [ ]:
device = "GPU" if torch.cuda.is_available() else "CPU"
print(f"YOLOv8s: {latency_ms(model):.1f} ms per image on the {device}")

## Exercises

1. Set `IMAGE` to a camera frame from GP1. How many objects does YOLOv8s report at
   0.25? Which of them are wrong, and why might a model trained on COCO photos make
   those mistakes?
2. In section 4, call `nms(candidates, per_class=False)`. Does the count change on the
   bus photo? Now picture a pedestrian standing half in front of a parked car. Explain
   why NMS across classes is dangerous there.
3. In section 3, decode the second most confident cell instead. Is it the same object?
4. Set `conf_cut = 0.001`, the value Ultralytics uses when it computes mAP. How many
   candidates are there now, and how many survive NMS?